# 01 — Problem Framing, Hypotheses & Data Ingestion: Ranking Signal Analysis

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 01 of 03 (Problem Framing, Hypotheses & Feature Vector Assembly)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for the data audit and feature engineering phase:
1. **Problem Framing & Operational Decision:** Define the operational content refresh decision, continuous search visibility target, and asymmetric error trade-offs.
2. **Core SEO Hypotheses:** Formulate and statistically test 4 core search hypotheses with explicit empirical verdicts (`CONFIRMED`, `MIXED`, `FALSE`, `OPPOSITE`).
3. **Data Ingestion & Scope Audit:** Verify atomic entity grain (`content_id`), document scope limitations, missingness patterns, and rate scaling.
4. **Feature Vector Assembly & Leakage Defense:** Implement SERP inverse power-law ranking signals, engagement ratios, missingness indicators, log-stabilizing transforms, enforce strict leakage blacklists, and export the validated feature dataset.

## 1. Problem Framing & Operational Decision

### 1.1 The Operational Decision
* **What operational decision does this model improve?**  
  Enterprise content and SEO teams manage portfolios spanning thousands of URLs across diverse client domains. Currently, content refresh prioritization relies on crude, lagging manual heuristics (e.g., sorting by calendar age or raw pageviews). This model predicts expected **search visibility** (the latent search impression capacity of a page based on its structural, semantic, and engagement signals). Content strategists and engineering teams act on this model to identify:
  1. **Under-indexed high-potential pages:** Content whose observed features warrant high search impressions but currently lag behind their predicted potential (prime refresh/optimization candidates).
  2. **Over-resourced low-potential pages:** Thin or poorly positioned content that should not consume scarce editorial rewrite hours.

* **Who acts on it?**  
  Content operations leads, technical SEO specialists, and client account managers.

### 1.2 Target Definition: Search Visibility
* **Why Continuous Regression instead of Binary Classification?**  
  Binary classification collapses search performance into an arbitrary threshold (e.g., "declining" vs. "stable"), obscuring orders of magnitude of search impact. Continuous regression models the true scale of visibility across the entire distribution.
* **Target Formulation:**
  $$\text{target\_log\_impressions} = \log(1 + \text{impressions\_90d})$$
  Raw search impressions follow an extreme power-law distribution spanning from single digits to hundreds of thousands. The $\log(1+x)$ transformation compresses extreme right-tail skewness into a well-behaved, symmetric distribution suitable for regression estimators.

### 1.3 Asymmetric Cost of Errors
* **Overestimation Error (False High Visibility):** The model predicts high visibility for low-yield content.  
  *Operational Cost:* Editorial teams waste expensive subject-matter expert hours rewriting, optimizing, and re-indexing pages that fail to generate organic traffic. Opportunity cost is high.
* **Underestimation Error (False Low Visibility):** The model predicts low visibility for high-yield content.  
  *Operational Cost:* The team neglects high-leverage assets, risking rank decay against aggressive competitor content.

## 2. Core SEO Hypotheses

Before building machine learning models, formulate four operational hypotheses based on prevailing SEO heuristics to test whether real-world data supports common industry beliefs:
1. **Hypothesis 1 (Content Depth):** *Article length (`word_count`) exhibits a strong positive relationship with search visibility.*
2. **Hypothesis 2 (Search Demand):** *Target keyword search volume (`search_volume`) strongly determines realized page impressions.*
3. **Hypothesis 3 (SERP Positioning):** *Average SERP rank position (`avg_position`) is inversely correlated with impressions (lower numerical rank = higher visibility).*
4. **Hypothesis 4 (Content Freshness):** *Recently updated content (`days_since_last_update` lower) commands higher search visibility than aged content.*

Each hypothesis should be evaluated using Spearman rank correlation ($\rho$), Pearson correlation ($r$), and visual distribution inspections, concluding with an explicit statistical verdict (`CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`).

In [ ]:
# Code Cell 1: Setup, Imports, and Data Ingestion
# TODO:
# 1. Import necessary libraries (numpy, pandas, matplotlib.pyplot, seaborn, scipy.stats).
# 2. Ingest 'data/raw/content_refresh_anonymized.csv'.
# 3. Assert 100% unique primary key grain on 'content_id' (should be 30,000 rows across 32 clients).
# 4. Construct the continuous regression target: target_log_impressions = np.log1p(df['impressions_90d']).
# 5. Display summary statistics of the target variable.


## 3. Data Ingestion: Scope & Limitations

### 3.1 Scope of the Dataset
* **Dataset File:** `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns).
* **Entity Grain:** Exactly 1 row per unique pseudonymized content item (`content_id`).
* **Client Representation:** Exactly 32 distinct pseudonymized enterprise client domains (`client_id`).
* **Observation Window:** A fixed trailing 90-day observation window ending at export time, with 30-day comparative windows (`*_last_30d` vs `*_prev_30d`).
* **Cohort Age:** Established, mature content only (`content_age_days >= 90`). Every row has at least 1 impression (`impressions_90d >= 1`).

### 3.2 Key Data Gotchas & Limitations
1. **Rate Scaling (×100):** Columns `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct` are recorded as ×100 percentages (`0.76` means `0.76%`, not `76%`).
2. **`avg_position = 0` Unranked State:** 1,205 rows have `avg_position = 0`, denoting unranked status / no impression position recorded, not rank zero.
3. **Systematic Missingness:** Keyword context is missing along `content_type` lines (e.g. `feedly article` rows have no keyword data; ~25.7% of rows lack `word_count`). Blind `fillna(0)` injects artificial category signals; missingness indicators are required.
4. **Observable Telemetry Only:** Internal product decision flags (`health_score`, `priority_score`, `action_type`) are deliberately excluded to prevent circular result leakage.

In [ ]:
# Code Cell 2: Completeness and Missingness Audit
# TODO:
# 1. Calculate missing value counts and percentages across all columns in df_raw.
# 2. Audit zero-value counts for numeric features.
# 3. Display the completeness summary table for columns with missing data.


In [ ]:
# Code Cell 3: Statistical Hypothesis Testing with Explicit Verdicts
# TODO:
# 1. Compute Spearman rho and Pearson r correlations for each hypothesis against target_log_impressions:
#    - H1: word_count
#    - H2: search_volume
#    - H3: avg_position (filter avg_position > 0)
#    - H4: days_since_last_update
# 2. Print a verdict summary table with explicit verdicts (CONFIRMED / MIXED / FALSE / OPPOSITE).
# 3. Generate a 2x2 grid of scatter plots illustrating each relationship and save to 'work/outputs/hypothesis_audit_distributions.png'.


### Interpretation of Hypotheses Findings
* **Guide for your write-up:**
  * Explain in plain words what each statistical verdict revealed about search traffic.
  * Why did H2 (search volume) show a mixed/weak correlation? (Broad head-terms face high competition).
  * Why was H3 (rank position) false when tested in isolation? (Position without search demand yields zero views).
  * Why did H4 (content freshness) show an opposite result? (Cumulative domain authority and backlink history outweigh recent date updates).

## 4. Feature Vector Assembly & Leakage Defense

### 4.1 Strict Leakage Blacklists
To ensure 100% scientific integrity, the feature pipeline strictly blacklists all leakers:
* **Pseudonymous Identifiers:** `content_id`, `client_id` (used solely for grouping and validation assertions).
* **Direct Target Telemetry:** `impressions_90d`, `clicks_90d`, `ctr` (since CTR contains impressions in its denominator).
* **Outcome Window Metrics:** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.
* **Target-Derived Labels:** `trend_direction`, `trend_pct`, `impression_tier`.

### 4.2 Domain-Specific Feature Engineering
Engineer high-leverage structural and behavioral signals:
1. **SERP Power-Law Inverse Rank (`serp_rank_power`):**
   $$\text{serp\_rank\_power} = \begin{cases} \frac{1}{\text{avg\_position}}, & \text{if } \text{avg\_position} > 0 \\ 0, & \text{otherwise} \end{cases}$$
2. **Rank Presence Indicator:** `has_rank_data = (avg_position > 0).astype(int)`.
3. **Rank Imputation:** `imputed_avg_position = np.where(avg_position > 0, avg_position, 100.0)`.
4. **Behavioral Ratios:**
   * `engaged_session_ratio = engaged_sessions_90d / (sessions_90d + 1)`
   * `scroll_per_pageview = scroll_events_90d / (pageviews_90d + 1)`
   * `ai_session_ratio = ai_sessions_90d / (sessions_90d + 1)`
5. **Demand vs. Position Synergy:** `demand_position_synergy = log_search_volume * serp_rank_power`.
6. **Engagement vs. Volume Synergy:** `engagement_volume_synergy = log_sessions_90d * serp_rank_power`.
7. **Systematic Missingness Indicators:** `missing_word_count`, `missing_search_volume`, `missing_competition`, `missing_cpc`.
8. **Log-Stabilizing Transforms:** Applied to heavy-tailed count columns (`word_count`, `search_volume`, `sessions_90d`, etc.).
9. **One-Hot Categorical Encoding:** Applied to `content_type`, `main_intent`, `competition_level`, `freshness_tier`, `position_tier`.

In [ ]:
# Code Cell 4: Feature Vector Assembly and Dataset Export
# TODO:
# 1. Apply feature engineering:
#    - Inverse SERP power-law features (serp_rank_power, log_avg_position, imputed_avg_position)
#    - Missingness indicator flags (missing_word_count, missing_search_volume, etc.)
#    - Behavioral engagement ratios (engaged_session_ratio, scroll_per_pageview, etc.)
#    - Log-stabilizing np.log1p() transforms on skewed count metrics
#    - Interaction synergy terms (demand_position_synergy, engagement_volume_synergy)
# 2. Apply pd.get_dummies() to categorical columns (content_type, main_intent, competition_level, freshness_tier, position_tier).
# 3. Concatenate metadata columns and all engineered numeric + dummy features.
# 4. Save the prepared dataset to:
#    - 'work/outputs/prepared_features.parquet'
#    - 'work/outputs/prepared_features.csv'
# 5. Print confirmation of matrix shape and assert zero target leakers in feature vector.
